# LeetCode 3966 — Count Good Integers in a Range bằng Python

- Link: https://leetcode.com/problems/count-good-integers-in-a-range/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động chữ số (digit DP), đếm theo tiền tố, đệ quy có nhớ, toán

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def goodIntegers(self, l: int, r: int, k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Một số nguyên được gọi là **tốt** nếu hiệu tuyệt đối của **mọi cặp chữ số kề nhau** không vượt quá `k`. Cho `l`, `r`, `k`, đếm số số tốt trong đoạn `[l, r]`. Vì `r` tới `10^15`, ta không thể duyệt từng số.

Đây là bài **digit DP** (DP theo chữ số) đúng nghĩa: điều kiện chỉ phụ thuộc vào cặp chữ số kề nhau, nên khi xây số từ trái sang phải ta chỉ cần nhớ **chữ số cuối cùng đã đặt**. Kết hợp với kỹ thuật "đếm số ≤ N" rồi trừ hai đầu: `đáp án = f(r) − f(l − 1)`.

Sau notebook này bạn sẽ:

1. Hiểu ý tưởng đếm số `≤ N` bằng cách "đi theo từng chữ số của `N`" và tách thành các nhánh "đã nhỏ hơn `N`".
2. Biết xây bảng `ways[độ dài][chữ số đầu]` bằng DP và dùng nó để đếm nhanh phần đuôi tự do.
3. Viết digit DP đệ quy có nhớ với trạng thái `(vị trí, chữ số trước, đang chạm giới hạn, đã bắt đầu)`.
4. Viết 4 lời giải (duyệt trực tiếp, sinh số tốt, digit DP có nhớ, đếm theo tiền tố với bảng `ways`) và đối chiếu bằng test ngẫu nhiên.
5. Phân tích `O(L · 10 · 10)` với `L ≤ 16` chữ số.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho ba số nguyên `l`, `r`, `k`. Số nguyên dương `x` là **tốt** nếu với mọi cặp chữ số liền kề `(a, b)` trong cách viết thập phân của `x` ta có `|a − b| ≤ k`. Số chỉ có một chữ số thì không có cặp nào nên luôn tốt. Đếm số số tốt trong `[l, r]`.

**Ví dụ 1:** `l = 10`, `r = 15`, `k = 1` → `3` (các số `10, 11, 12`: `|1−0| = 1`, `|1−1| = 0`, `|1−2| = 1`; còn `13` có `|1−3| = 2 > 1`).

**Ví dụ 2:** `l = 201`, `r = 204`, `k = 2` → `2` (`201` và `202`; `203` có `|0−3| = 3 > 2`).

**Ràng buộc**

- `10 <= l <= r <= 10^15`
- `0 <= k <= 9`

Lưu ý: `k = 9` nghĩa mọi số đều tốt (hiệu hai chữ số luôn `≤ 9`) nên đáp án là `r − l + 1`; `k = 0` nghĩa mọi chữ số trong số phải **giống nhau** (`11, 222, 7777...`).

In [1]:
examples = [
    (dict(l=10, r=15, k=1), 3),
    (dict(l=201, r=204, k=2), 2),
]
for args, expected in examples:
    print("l =", args["l"], "| r =", args["r"], "| k =", args["k"], "| mong đợi:", expected)

l = 10 | r = 15 | k = 1 | mong đợi: 3
l = 201 | r = 204 | k = 2 | mong đợi: 2


## 3. Kiến thức nền

### 3.1. Chữ số của một số: chuyển sang chuỗi và `int(ch)`

Cách đơn giản nhất để lấy các chữ số của `N` là `str(N)`, rồi đọc từng ký tự và `int(...)` nếu cần số.

In [2]:
N = 40721
s = str(N)
print(s, len(s))                      # '40721' 5
digits = [int(ch) for ch in s]
print(digits)                         # [4, 0, 7, 2, 1]
print([abs(a - b) for a, b in zip(digits, digits[1:])])   # hiệu tuyệt đối các cặp kề: [4, 7, 5, 1]

40721 5
[4, 0, 7, 2, 1]
[4, 7, 5, 1]


### 3.2. Hàm `is_good` và duyệt vét cạn

Một số là tốt nếu mọi hiệu `|a − b|` giữa các chữ số kề nhau đều `≤ k`. `all(...)` trả `True` khi mọi phần tử đúng (và `True` với dãy rỗng — đúng ý "số một chữ số luôn tốt").

In [3]:
def is_good(x, k):
    d = [int(c) for c in str(x)]
    return all(abs(a - b) <= k for a, b in zip(d, d[1:]))

print([x for x in range(10, 30) if is_good(x, 1)])     # 10, 11, 12, 21, 22, 23
print(is_good(7, 0), is_good(77, 0), is_good(78, 0))   # True True False

[10, 11, 12, 21, 22, 23]
True True False


### 3.3. Ý tưởng "đếm `≤ N`" rồi trừ hai đầu

Gọi `f(N)` = số số tốt trong `[1, N]`. Khi đó số số tốt trong `[l, r]` là `f(r) − f(l − 1)`. Điều này đưa bài toán khoảng về hai bài toán "đếm đến `N`", đơn giản hơn nhiều.

In [4]:
def f_brute(N, k):
    return sum(1 for x in range(1, N + 1) if is_good(x, k))

l, r, k = 10, 15, 1
print(f_brute(r, k) - f_brute(l - 1, k))     # 3

3


### 3.4. Tiền tố "chạm giới hạn" (tight)

Khi xây số từ trái sang phải để được số `≤ N` có cùng độ dài, tại mỗi vị trí có hai tình huống:

- **chưa nhỏ hơn:** các chữ số đã đặt trùng khít với `N` — chữ số kế tiếp bị giới hạn `≤ N[i]` (trạng thái "tight");
- **đã nhỏ hơn:** đã có một vị trí chữ số nhỏ hơn `N` — các chữ số sau **tự do** (`0..9`), số chắc chắn `< N`.

Chia thành các nhánh "chọn chữ số nhỏ hơn `N[i]` tại vị trí `i`, rồi phần đuôi tự do" cho phép đếm phần đuôi bằng một bảng tính sẵn.

In [5]:
N = "4072"
for i, ch in enumerate(N):
    print(f"vị trí {i}: giữ nguyên tiền tố '{N[:i]}', chọn chữ số nhỏ hơn '{ch}' -> {list(range(0 if i else 1, int(ch)))}, đuôi {len(N) - i - 1} chữ số tự do")

vị trí 0: giữ nguyên tiền tố '', chọn chữ số nhỏ hơn '4' -> [1, 2, 3], đuôi 3 chữ số tự do
vị trí 1: giữ nguyên tiền tố '4', chọn chữ số nhỏ hơn '0' -> [], đuôi 2 chữ số tự do
vị trí 2: giữ nguyên tiền tố '40', chọn chữ số nhỏ hơn '7' -> [0, 1, 2, 3, 4, 5, 6], đuôi 1 chữ số tự do
vị trí 3: giữ nguyên tiền tố '407', chọn chữ số nhỏ hơn '2' -> [0, 1], đuôi 0 chữ số tự do


### 3.5. Đệ quy có nhớ `lru_cache`

`functools.lru_cache` lưu kết quả của hàm theo tham số, nên các lời gọi lặp (cùng trạng thái) chỉ tính một lần. Điều kiện: tham số phải băm được (số, tuple...).

In [6]:
from functools import lru_cache

calls = {"n": 0}
@lru_cache(maxsize=None)
def fib(n):
    calls["n"] += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(40), "- số lần tính thực sự:", calls["n"])      # 102334155 - 41

102334155 - số lần tính thực sự: 41


## 4. Ý tưởng nền tảng / trực giác

Tính tay `f(15)` và `f(9)` với `k = 1`:

- `f(9)`: các số `1..9`, mỗi số một chữ số nên đều tốt ⇒ `9`.
- `f(15)`: ngoài `1..9`, thêm các số hai chữ số `10..15`. `10 ✓`, `11 ✓`, `12 ✓`, `13 ✗`, `14 ✗`, `15 ✗` ⇒ `9 + 3 = 12`.

Đáp án `f(15) − f(9) = 3` ✓.

Với `r` lớn không thể liệt kê. Ta cần đếm "số tốt có độ dài `L` và `≤ N`" mà không liệt kê. Ý tưởng: nhóm các số theo **tiền tố giống `N`** đến vị trí `i`, rồi chọn chữ số nhỏ hơn `N[i]`; phần còn lại là **bất kỳ dãy chữ số nào** miễn là mọi cặp kề thỏa. Số dãy như vậy chỉ phụ thuộc **độ dài còn lại** và **chữ số đầu của phần đuôi** ⇒ một bảng nhỏ `ways[len][d]` đủ dùng.

## 5. Cách 1 — Vét cạn: duyệt mọi số trong `[l, r]`

**Ý tưởng.** Với mỗi `x` từ `l` đến `r`, kiểm tra `is_good(x, k)`. Dùng cho `r − l` nhỏ để đối chiếu.

In [7]:
def brute_force(l, r, k):
    return sum(1 for x in range(l, r + 1) if is_good(x, k))

for args, expected in examples:
    got = brute_force(**args)
    print(got, expected)
    assert got == expected

3 3
2 2


**Độ phức tạp.** `O((r − l + 1) · log r)`. Với `r` tới `10^15` là bất khả thi. Chỉ dùng để kiểm chứng.

### 5.1. Cách 1b — Sinh các số tốt bằng DFS (kiểm chứng độc lập)

Thay vì duyệt mọi số, ta **sinh** các số tốt: đặt chữ số đầu `1..9`, rồi mỗi bước thêm chữ số `d` với `|d − chữ số cuối| ≤ k`. Dừng khi vượt `N`. Số lượng số tốt có thể ít hơn nhiều so với `N` khi `k` nhỏ, nhưng khi `k` lớn vẫn rất nhiều (`k = 9`: mọi số).

In [8]:
def count_by_generation(N, k):
    """Số số tốt trong [1, N] bằng cách sinh DFS (chỉ dùng cho N vừa)."""
    if N <= 0:
        return 0
    total = 0
    stack = [(d, d) for d in range(1, 10) if d <= N]       # (giá trị, chữ số cuối)
    while stack:
        x, last = stack.pop()
        total += 1
        for d in range(max(0, last - k), min(9, last + k) + 1):
            y = x * 10 + d
            if y <= N:
                stack.append((y, d))
    return total

for N in [9, 15, 99, 1000]:
    assert count_by_generation(N, 1) == f_brute(N, 1)
print("sinh DFS khớp vét cạn; f(1000) với k=1:", count_by_generation(1000, 1))

sinh DFS khớp vét cạn; f(1000) với k=1: 111


**Nút thắt.** Nhiều tiền tố khác nhau dẫn tới cùng trạng thái "chữ số cuối là `d`, còn `m` chữ số nữa" và có **cùng số cách hoàn thành**. Ta chỉ cần đếm số cách hoàn thành một lần cho mỗi cặp `(m, d)`.

## 6. Bước đột phá

**Insight 1 — chỉ cần nhớ chữ số cuối.** Điều kiện tốt chỉ ràng buộc cặp kề, nên số cách hoàn thành phần đuôi chỉ phụ thuộc chữ số cuối đã đặt.

**Insight 2 — bảng `ways`.** Định nghĩa `ways[m][d]` = số xâu chữ số độ dài `m` **bắt đầu bằng `d`** (chữ số đầu là `d`) mà mọi cặp kề đều `≤ k`:

- `ways[1][d] = 1`;
- `ways[m][d] = Σ_{e : |d − e| ≤ k} ways[m − 1][e]`.

Đây là phép nhân ma trận kề `10 × 10` lặp lại; với `m ≤ 16` ta chỉ cần `16 · 10 · 10` phép cộng.

**Insight 3 — đếm `≤ N` theo tiền tố.** Với `N` có `L` chữ số:

- Mọi số có **ít chữ số hơn** (độ dài `ℓ = 1..L−1`): chữ số đầu `1..9` bất kỳ: `Σ_{d=1}^{9} ways[ℓ][d]`.
- Số có **đúng `L` chữ số** và `≤ N`: duyệt `i = 0..L−1`; tại `i`, chọn chữ số `d < N[i]` (với `i = 0` thì `d ≥ 1`) sao cho `d` tương thích với chữ số trước `prev` (`|d − prev| ≤ k`, nếu `i > 0`); cộng `ways[L − i][d]`. Rồi kiểm tra `N[i]` có tương thích với `prev` không; nếu không thì **dừng** (mọi số bắt đầu bằng tiền tố `N[0..i]` đều không tốt); nếu tương thích thì đặt `prev = N[i]` và tiếp tục. Nếu đi hết `L` chữ số mà không dừng, chính `N` là số tốt: cộng `1`.

**Insight 4 — hai đầu.** Đáp án `= f(r) − f(l − 1)`.

**Quy tắc "nếu… thì…":**

- Nếu `N[i]` không tương thích với chữ số trước thì không số nào có tiền tố `N[0..i]` là tốt, **dừng** vòng lặp.
- Nếu `d` không tương thích với `prev` thì bỏ qua `d` đó.
- `i = 0` bắt buộc `d ≥ 1` (không cho số bắt đầu bằng `0`); các độ dài ngắn hơn đã xử lý riêng.
- `f(0) = 0` (không có số dương).

## 7. Các cách tiếp theo

### 7.1. Cách 3 — Digit DP đệ quy có nhớ với trạng thái `(pos, prev, tight, started)`

**Giải thích từng đoạn.**

1. Biểu diễn `N` thành danh sách chữ số `s` (độ dài `L`).
2. `go(pos, prev, tight, started)` trả về số cách điền các chữ số từ vị trí `pos` tới cuối:
   - `prev`: chữ số cuối đã đặt (`-1` nếu chưa đặt số khác `0` đầu tiên);
   - `tight`: các chữ số đã đặt trùng khít tiền tố `N` (chữ số kế bị chặn `≤ s[pos]`);
   - `started`: đã đặt chữ số khác `0` đầu tiên chưa (để cho phép "số ngắn hơn" bằng các số `0` ở đầu).
3. Khi `pos == L`: nếu `started` thì đếm `1` (đây là một số dương hợp lệ), ngược lại `0` (toàn `0`, không phải số dương).
4. Với mỗi chữ số `d` từ `0` tới `s[pos]` (nếu `tight`) hoặc `9`:
   - nếu `started` và `|d − prev| > k` ⇒ bỏ qua;
   - nếu chưa bắt đầu và `d = 0` ⇒ vẫn chưa bắt đầu (`started` giữ `False`, `prev = -1`);
   - ngược lại gọi đệ quy với `started = True` và `prev = d`.

`lru_cache` lưu theo `(pos, prev, tight, started)`: tối đa `16 · 11 · 2 · 2` trạng thái.

In [9]:
from functools import lru_cache

def count_le_memo(N, k):
    """Số số tốt trong [1, N] bằng digit DP đệ quy có nhớ."""
    if N <= 0:
        return 0
    s = [int(c) for c in str(N)]
    L = len(s)

    @lru_cache(maxsize=None)
    def go(pos, prev, tight, started):
        if pos == L:
            return 1 if started else 0
        limit = s[pos] if tight else 9
        total = 0
        for d in range(limit + 1):
            if started and abs(d - prev) > k:
                continue
            if not started and d == 0:
                total += go(pos + 1, -1, tight and d == limit, False)
            else:
                total += go(pos + 1, d, tight and d == limit, True)
        return total

    return go(0, -1, True, False)

def with_memo(l, r, k):
    return count_le_memo(r, k) - count_le_memo(l - 1, k)

for args, expected in examples:
    got = with_memo(**args)
    print(got, expected)
    assert got == expected

3 3
2 2


**Độ phức tạp.** Số trạng thái `O(L · 11 · 2 · 2)`, mỗi trạng thái thử `≤ 10` chữ số ⇒ `O(L · 10 · 11 · 4)` ≈ `7000` phép; thực tế rất nhanh. Bộ nhớ `O(L · 44)`.

### 7.2. Cách 4 — Đếm theo tiền tố với bảng `ways` (bản chính, gọn)

**Giải thích từng đoạn.**

1. Dựng `ways[m][d]` cho `m = 1..L` (L = số chữ số của `r`, tối đa 16).
2. `count_le(N)`: 
   - cộng `Σ_{ℓ<L} Σ_{d=1..9} ways[ℓ][d]` (mọi số ngắn hơn);
   - duyệt vị trí `i`; với mỗi `d < N[i]` hợp lệ (`d ≥ 1` nếu `i = 0`, và `|d − prev| ≤ k` nếu `i > 0`) cộng `ways[L − i][d]`;
   - kiểm tra `N[i]` với `prev`; nếu vi phạm thì dừng; hết vòng không vi phạm thì `+1` (chính `N`).
3. `đáp án = count_le(r) − count_le(l − 1)`.

In [10]:
def build_ways(k, maxlen):
    """ways[m][d]: số xâu chữ số độ dài m, bắt đầu bằng d, mọi cặp kề có |hiệu| <= k."""
    ways = [[0] * 10 for _ in range(maxlen + 1)]
    for d in range(10):
        ways[1][d] = 1
    for m in range(2, maxlen + 1):
        for d in range(10):
            ways[m][d] = sum(ways[m - 1][e] for e in range(10) if abs(d - e) <= k)
    return ways

def count_le_prefix(N, k, ways):
    if N <= 0:
        return 0
    s = [int(c) for c in str(N)]
    L = len(s)
    total = 0
    for length in range(1, L):                       # mọi số có ít chữ số hơn N
        total += sum(ways[length][d] for d in range(1, 10))
    prev = -1
    ok = True
    for i in range(L):
        for d in range(1 if i == 0 else 0, s[i]):    # chữ số nhỏ hơn N[i] tại vị trí i
            if i == 0 or abs(d - prev) <= k:
                total += ways[L - i][d]
        if i > 0 and abs(s[i] - prev) > k:           # tiền tố của N không còn tốt
            ok = False
            break
        prev = s[i]
    if ok:
        total += 1                                   # chính N là số tốt
    return total

def with_prefix(l, r, k):
    ways = build_ways(k, len(str(r)))
    return count_le_prefix(r, k, ways) - count_le_prefix(l - 1, k, ways)

for args, expected in examples:
    got = with_prefix(**args)
    print(got, expected)
    assert got == expected

3 3
2 2


**Độ phức tạp.** Dựng bảng `O(L · 10 · 10)`; mỗi `count_le_prefix` `O(L · 10)`. Tổng `O(L · 100)` với `L ≤ 16` — vài trăm phép tính, tức thì.

## 8. TRACE từng bước

Ta trace `count_le_prefix(15, k=1)` rồi `count_le_prefix(9, k=1)`. In bảng `ways` ban đầu (với `k = 1`), các số ngắn hơn, và từng bước tại mỗi vị trí chữ số.

Bảng mong đợi cho `N = 15`, `k = 1`:

| Bước | Hành động | Cộng |
|---|---|---|
| số ngắn hơn (độ dài 1) | `Σ_{d=1..9} ways[1][d]` | `9` |
| `i = 0`, `N[0] = 1` | chọn `d < 1` với `d ≥ 1`: không có | `0` |
| kiểm tra `N[0]` | `i = 0`, không có `prev` ⇒ đặt `prev = 1` | — |
| `i = 1`, `N[1] = 5` | chọn `d ∈ {0,..,4}` có `|d − 1| ≤ 1`: `d = 0, 1, 2`; mỗi `ways[1][d] = 1` | `3` |
| kiểm tra `N[1]` | `|5 − 1| = 4 > 1` ⇒ dừng (`N = 15` không tốt) | — |
| tổng | `9 + 3` | `12` |

In [11]:
def trace_count_le(N, k):
    s = [int(c) for c in str(N)]
    L = len(s)
    ways = build_ways(k, L)
    print(f"N = {N}, k = {k}, chữ số = {s}")
    print("ways[m][d] (m=1..%d):" % L)
    for m in range(1, L + 1):
        print("  m=%d:" % m, ways[m])
    total = 0
    for length in range(1, L):
        add = sum(ways[length][d] for d in range(1, 10))
        total += add
        print(f"số có {length} chữ số (mọi chữ số đầu 1..9): +{add} -> {total}")
    prev, ok = -1, True
    for i in range(L):
        cands = [d for d in range(1 if i == 0 else 0, s[i]) if i == 0 or abs(d - prev) <= k]
        add = sum(ways[L - i][d] for d in cands)
        total += add
        print(f"i={i}, N[i]={s[i]}, prev={prev}: chọn d<{s[i]} hợp lệ {cands}, +{add} (đuôi {L-i} chữ số) -> {total}")
        if i > 0 and abs(s[i] - prev) > k:
            print(f"   |{s[i]} - {prev}| = {abs(s[i]-prev)} > {k}: tiền tố không tốt -> dừng")
            ok = False
            break
        prev = s[i]
    if ok:
        total += 1
        print("N chính nó là số tốt: +1 ->", total)
    print("f(%d) =" % N, total)
    return total

trace_count_le(15, 1)
trace_count_le(9, 1)

N = 15, k = 1, chữ số = [1, 5]
ways[m][d] (m=1..2):
  m=1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  m=2: [2, 3, 3, 3, 3, 3, 3, 3, 3, 2]
số có 1 chữ số (mọi chữ số đầu 1..9): +9 -> 9
i=0, N[i]=1, prev=-1: chọn d<1 hợp lệ [], +0 (đuôi 2 chữ số) -> 9
i=1, N[i]=5, prev=1: chọn d<5 hợp lệ [0, 1, 2], +3 (đuôi 1 chữ số) -> 12
   |5 - 1| = 4 > 1: tiền tố không tốt -> dừng
f(15) = 12
N = 9, k = 1, chữ số = [9]
ways[m][d] (m=1..1):
  m=1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
i=0, N[i]=9, prev=-1: chọn d<9 hợp lệ [1, 2, 3, 4, 5, 6, 7, 8], +8 (đuôi 1 chữ số) -> 8
N chính nó là số tốt: +1 -> 9
f(9) = 9


9

In [12]:
# Trace khi N là số tốt: đi hết các chữ số không dừng và cộng 1 cho chính N
trace_count_le(212, 1)
print("vét cạn f(212):", f_brute(212, 1))

N = 212, k = 1, chữ số = [2, 1, 2]
ways[m][d] (m=1..3):
  m=1: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  m=2: [2, 3, 3, 3, 3, 3, 3, 3, 3, 2]
  m=3: [5, 8, 9, 9, 9, 9, 9, 9, 8, 5]
số có 1 chữ số (mọi chữ số đầu 1..9): +9 -> 9
số có 2 chữ số (mọi chữ số đầu 1..9): +26 -> 35
i=0, N[i]=2, prev=-1: chọn d<2 hợp lệ [1], +8 (đuôi 3 chữ số) -> 43
i=1, N[i]=1, prev=2: chọn d<1 hợp lệ [], +0 (đuôi 2 chữ số) -> 43
i=2, N[i]=2, prev=1: chọn d<2 hợp lệ [0, 1], +2 (đuôi 1 chữ số) -> 45
N chính nó là số tốt: +1 -> 46
f(212) = 46
vét cạn f(212): 46


Ở `N = 212` với `k = 1`, mọi cặp kề `(2,1)` và `(1,2)` đều thỏa nên vòng lặp không dừng và cộng thêm `1` cho chính `212`. Kết quả khớp vét cạn.

## 9. Code nộp LeetCode

Dùng cách 4 (đếm theo tiền tố với bảng `ways`), viết gọn trong một phương thức với hàm phụ.

In [13]:
class Solution:
    def goodIntegers(self, l: int, r: int, k: int) -> int:
        L = len(str(r))
        # ways[m][d]: số xâu chữ số độ dài m bắt đầu bằng d, mọi cặp kề có |hiệu| <= k
        ways = [[0] * 10 for _ in range(L + 1)]
        for d in range(10):
            ways[1][d] = 1
        for m in range(2, L + 1):
            for d in range(10):
                ways[m][d] = sum(ways[m - 1][e] for e in range(10) if abs(d - e) <= k)

        def count_le(N: int) -> int:                 # số số tốt trong [1, N]
            if N <= 0:
                return 0
            s = [int(c) for c in str(N)]
            n = len(s)
            total = 0
            for length in range(1, n):               # các số ngắn hơn N
                total += sum(ways[length][d] for d in range(1, 10))
            prev = -1
            for i in range(n):
                for d in range(1 if i == 0 else 0, s[i]):
                    if i == 0 or abs(d - prev) <= k:
                        total += ways[n - i][d]
                if i > 0 and abs(s[i] - prev) > k:   # tiền tố của N đã không tốt
                    return total
                prev = s[i]
            return total + 1                         # N là số tốt

        return count_le(r) - count_le(l - 1)


sol = Solution()
for args, expected in examples:
    got = sol.goodIntegers(**args)
    print(got, expected)
    assert got == expected

3 3
2 2


## 10. Vì sao thuật toán đúng

**Chia khoảng.** Các số tốt trong `[l, r]` là các số tốt trong `[1, r]` bỏ đi các số tốt trong `[1, l − 1]`; hiệu `f(r) − f(l − 1)` đếm đúng (vì `l ≥ 10` nên `l − 1 ≥ 9 ≥ 1`).

**Phân loại số `≤ N` theo độ dài và vị trí khác biệt đầu tiên.**

- Số có ít chữ số hơn `N`: đều `< N`; mỗi độ dài `ℓ` và chữ số đầu `d ∈ 1..9` cho `ways[ℓ][d]` số tốt (đúng theo định nghĩa `ways`).
- Số có đúng `L` chữ số `< N`: tồn tại **vị trí đầu tiên `i`** mà chữ số của nó nhỏ hơn `N[i]`, các vị trí trước giống `N`. Hai số khác nhau ở vị trí `i` hoặc ở tiền tố khác nhau không bị đếm trùng. Điều kiện số đó tốt = tiền tố `N[0..i−1]` tốt, cặp `(N[i−1], d)` thỏa, và phần đuôi bắt đầu từ `d` thỏa: số cách chính là `ways[L − i][d]` (xâu độ dài `L − i` bắt đầu bằng `d`).
- Nếu tiền tố `N[0..i]` đã chứa cặp xấu thì mọi số có tiền tố này đều không tốt, nên không cần xét `i` lớn hơn: **dừng** là đúng.
- Số bằng `N`: tốt khi toàn bộ chữ số thỏa — đúng khi vòng lặp chạy hết mà không dừng.

**Bảng `ways` đúng** theo quy nạp theo `m`: một xâu độ dài `m` bắt đầu bằng `d` gồm `d` và một xâu độ dài `m − 1` bắt đầu bằng `e` thỏa `|d − e| ≤ k`.

**Cách 3 (có nhớ) tương đương:** `go` duyệt mọi chữ số tại mỗi vị trí, trạng thái `(tight, started)` mã hóa đúng "còn trùng tiền tố `N`" và "đã đặt chữ số khác `0` đầu tiên" nên mọi số `≤ N` (kể cả ngắn hơn, viết với `0` đứng đầu) được đếm đúng một lần.

## 11. Phân tích Time/Space

Đặt `L` = số chữ số của `r` (`L ≤ 16`).

| Cách | Time | Space |
|---|---|---|
| Vét cạn | `O((r − l) · L)` | `O(1)` |
| Sinh DFS | `O(số số tốt ≤ N)` | `O(L)` stack theo nhánh / `O(số số tốt)` |
| Digit DP có nhớ | `O(L · 11 · 4 · 10)` | `O(L · 44)` |
| Đếm theo tiền tố (nộp) | `O(L · 10 · 10)` | `O(L · 10)` |

Với giới hạn đề, các cách DP chạy trong dưới 1 mili-giây (cực nhanh).

In [14]:
import time
t0 = time.time()
res = Solution().goodIntegers(10, 10**15, 3)
print("goodIntegers(10, 1e15, 3) =", res, f"- {time.time()-t0:.5f}s")
print("k=9 mọi số đều tốt:", Solution().goodIntegers(10, 10**15, 9), "=", 10**15 - 10 + 1)
print("k=0 chỉ các số lặp chữ số:", Solution().goodIntegers(10, 10**15, 0))
# kiểm tra k=0: có 9 số mỗi độ dài 2..15 (11..99, 111..999, ...) và 10^15 (16 chữ số) không tốt: 9 * 14
assert Solution().goodIntegers(10, 10**15, 0) == 9 * 14

goodIntegers(10, 1e15, 3) = 1045029672177 - 0.00019s
k=9 mọi số đều tốt: 999999999999991 = 999999999999991
k=0 chỉ các số lặp chữ số: 126


## 12. Bộ test

Chiến lược test:

1. 2 ví dụ của đề (assert).
2. Biên: `k = 9` (mọi số tốt) và `k = 0` (mọi chữ số giống nhau); `l = r`; `l`, `r` đúng bằng lũy thừa của 10 (`10`, `100`, `10^15`).
3. Phản ví dụ: không dừng khi tiền tố xấu, hoặc cho phép chữ số `0` đứng đầu (phần 13).
4. Đối chiếu vét cạn, sinh DFS, digit DP có nhớ, đếm theo tiền tố và `Solution` trên nhiều đoạn ngẫu nhiên (seed cố định).

In [15]:
sol = Solution()
for args, expected in examples:
    assert sol.goodIntegers(**args) == expected

assert sol.goodIntegers(10, 99, 9) == 90                # mọi số hai chữ số đều tốt khi k = 9
assert sol.goodIntegers(10, 99, 0) == 9                 # 11, 22, ..., 99
assert sol.goodIntegers(11, 11, 0) == 1 and sol.goodIntegers(12, 12, 0) == 0
assert sol.goodIntegers(100, 100, 1) == 1               # |1-0| = 1 <= 1 và |0-0| = 0 <= 1 -> 100 tốt với k = 1
assert sol.goodIntegers(100, 100, 0) == 0               # với k = 0 thì 100 không tốt (|1-0| = 1 > 0)
assert sol.goodIntegers(10**15, 10**15, 0) == 0
assert sol.goodIntegers(10**15, 10**15, 1) == 1         # 1 và 15 chữ số 0: mọi cặp chênh <= 1
print("test cố định OK")

test cố định OK


In [16]:
import random
rng = random.Random(20260101)
cnt = 0
for _ in range(1500):
    l = rng.randint(10, 3000)
    r = rng.randint(l, l + rng.choice([0, 5, 50, 500, 3000]))
    k = rng.randint(0, 9)
    expected = brute_force(l, r, k)
    assert with_memo(l, r, k) == expected, (l, r, k)
    assert with_prefix(l, r, k) == expected, (l, r, k)
    assert Solution().goodIntegers(l, r, k) == expected, (l, r, k)
    assert count_by_generation(r, k) - count_by_generation(l - 1, k) == expected, (l, r, k)
    cnt += 1
print("đã đối chiếu", cnt, "đoạn ngẫu nhiên: tất cả khớp vét cạn và sinh DFS")

đã đối chiếu 1500 đoạn ngẫu nhiên: tất cả khớp vét cạn và sinh DFS


In [17]:
# Số lớn: so digit DP có nhớ với đếm theo tiền tố và với sinh DFS (k nhỏ để số số tốt không quá nhiều)
rng = random.Random(555)
for _ in range(200):
    r = rng.randint(10, 10**15)
    l = rng.randint(10, r)
    k = rng.randint(0, 9)
    assert with_memo(l, r, k) == with_prefix(l, r, k) == Solution().goodIntegers(l, r, k)
for N, k in [(10**6, 1), (10**6, 2), (10**6, 3), (123456, 0), (999999, 4)]:
    assert count_by_generation(N, k) == count_le_prefix(N, k, build_ways(k, 7)), (N, k)
print("hai cách DP khớp trên 200 đoạn lớn; sinh DFS khớp ở các N tới 10^6")

hai cách DP khớp trên 200 đoạn lớn; sinh DFS khớp ở các N tới 10^6


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không dừng khi tiền tố của `N` đã vi phạm

**Code sai:** bỏ dòng "nếu `|N[i] − prev| > k` thì dừng".
**Vì sao sai:** các vị trí sau vẫn cộng thêm các số có tiền tố giống `N` — tiền tố đã có cặp xấu nên không có số nào tốt; kết quả bị đếm thừa.
**Code đúng:** `if abs(s[i] - prev) > k: return total` (hoặc `break`).

In [18]:
def wrong_no_stop(N, k, ways):
    s = [int(c) for c in str(N)]
    L = len(s)
    total = 0
    for length in range(1, L):
        total += sum(ways[length][d] for d in range(1, 10))
    prev = -1
    for i in range(L):
        for d in range(1 if i == 0 else 0, s[i]):
            if i == 0 or abs(d - prev) <= k:
                total += ways[L - i][d]
        prev = s[i]                                   # SAI: không kiểm tra / không dừng
    return total + 1

ways1 = build_ways(1, 3)
print("sai :", wrong_no_stop(15, 1, ways1), " đúng:", f_brute(15, 1))

sai : 13  đúng: 12


### Lỗi 2 — Cho phép chữ số `0` đứng đầu số

**Code sai:** ở `i = 0` vẫn duyệt `d` từ `0`.
**Vì sao sai:** `d = 0` ở đầu tương ứng "số có ít chữ số hơn" đã được đếm riêng ở phần độ dài ngắn hơn ⇒ đếm trùng (và trong đó `ways[..][0]` coi `0` là chữ số thật).
**Code đúng:** `range(1 if i == 0 else 0, s[i])`.

In [19]:
def wrong_leading_zero(N, k, ways):
    s = [int(c) for c in str(N)]
    L = len(s)
    total = 0
    for length in range(1, L):
        total += sum(ways[length][d] for d in range(1, 10))
    prev = -1
    for i in range(L):
        for d in range(0, s[i]):                      # SAI: i = 0 cũng cho d = 0
            if i == 0 or abs(d - prev) <= k:
                total += ways[L - i][d]
        if i > 0 and abs(s[i] - prev) > k:
            return total
        prev = s[i]
    return total + 1

print("sai :", wrong_leading_zero(15, 1, ways1), " đúng:", f_brute(15, 1))

sai : 14  đúng: 12


### Lỗi 3 — Quên cộng chính `N` hoặc cộng khi `N` không tốt

**Code sai:** `return total` (thiếu `+ 1`) hoặc luôn `+ 1`.
**Vì sao sai:** bài đếm `≤ N` (bao gồm `N`). Chỉ cộng `1` khi toàn bộ các cặp kề của `N` thỏa (vòng lặp chạy hết).
**Code đúng:** `return total + 1` ở cuối (sau vòng lặp không dừng).

In [20]:
def wrong_forget_N(N, k, ways):
    s = [int(c) for c in str(N)]
    L = len(s)
    total = 0
    for length in range(1, L):
        total += sum(ways[length][d] for d in range(1, 10))
    prev = -1
    for i in range(L):
        for d in range(1 if i == 0 else 0, s[i]):
            if i == 0 or abs(d - prev) <= k:
                total += ways[L - i][d]
        if i > 0 and abs(s[i] - prev) > k:
            return total
        prev = s[i]
    return total                                      # SAI: thiếu + 1 cho chính N

print("sai :", wrong_forget_N(212, 1, build_ways(1, 3)), " đúng:", f_brute(212, 1))

sai : 45  đúng: 46


### Lỗi 4 — Tính `f(l)` thay vì `f(l − 1)`

**Code sai:** `count_le(r) - count_le(l)`.
**Vì sao sai:** loại mất chính `l` nếu `l` là số tốt (đoạn là `[l, r]` đóng).
**Code đúng:** `count_le(r) - count_le(l - 1)`.

### Lỗi 5 — Lặp `k = 0` hay `k = 9` bị xử lý như trường hợp đặc biệt rồi sai

**Mẹo:** không cần ca đặc biệt; công thức `|d − e| ≤ k` tự cho `k = 9` thành "mọi cặp" và `k = 0` thành "bằng nhau". Hãy kiểm tra bằng assert như ở phần 11.

## 14. Mẹo & nhận dạng pattern

- **"Đếm các số trong `[l, r]` thỏa điều kiện trên các chữ số"** với `r` rất lớn → digit DP: `f(r) − f(l − 1)`.
- Trạng thái nên chứa đúng thông tin tương lai cần: ở đây là **chữ số cuối**; thêm cờ `tight` và `started` nếu dùng đệ quy.
- Có hai phong cách: **(A)** đệ quy có nhớ với `tight/started`; **(B)** đếm theo tiền tố ("tại vị trí `i` chọn chữ số nhỏ hơn") với bảng `ways` — cách (B) ngắn gọn, ít lỗi hơn khi điều kiện chỉ phụ thuộc cặp kề.
- Khi tiền tố của `N` đã vi phạm điều kiện, các nhánh sau của `N` bị loại (nhớ `break`).
- Luôn kiểm chứng bằng vét cạn trên đoạn nhỏ ngẫu nhiên.
- Các bài cùng họ: *Numbers With Repeated Digits*, *Count Stepping Numbers in Range*, *Digit Count in Range*.

## 15. Biến thể

1. **Stepping numbers** (mọi cặp kề chênh đúng `1`): đổi điều kiện `|d − e| ≤ k` thành `|d − e| == 1` (Bài 5).
2. **Chữ số kề phải khác nhau** (`|d − e| ≥ 1`).
3. **Tổng chữ số bằng `S`** hoặc chia hết cho `S`: trạng thái là (tổng, số dư) thay vì chữ số cuối.
4. **Điều kiện trên cặp không kề** (khoảng cách 2): nhớ hai chữ số cuối.
5. **Cơ số khác 10:** thay `10` bằng cơ số `b`, bảng `ways` kích thước `b`.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | kiểm tra từng số | `O((r−l)·L)` | `O(1)` | đối chiếu đoạn nhỏ |
| 1b. Sinh DFS | sinh trực tiếp số tốt `≤ N` | `O(số số tốt)` | `O(số số tốt)` | đối chiếu `N` vừa, `k` nhỏ |
| 3. Digit DP có nhớ | `(pos, prev, tight, started)` | `O(L·10·44)` | `O(L·44)` | mẫu tổng quát |
| 4. Đếm theo tiền tố + `ways` | chia theo vị trí khác biệt đầu tiên | `O(L·100)` | `O(L·10)` | **nộp bài** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** Với `k = 1`, `build_ways(1, 2)[2][3]` bằng bao nhiêu? (số xâu hai chữ số bắt đầu bằng `3` có hiệu `≤ 1`)

**Bài 2 (tính tay).** Đếm số tốt trong `[10, 20]` khi `k = 0`.

**Bài 3 (điền TODO).** Hoàn thiện `my_build_ways(k, maxlen)`.

**Bài 4 (tự viết lại).** Viết `my_count_le(N, k)` đếm số tốt trong `[1, N]` theo cách tiền tố (có thể dùng `build_ways`).

**Bài 5 (biến thể).** Viết `count_stepping(l, r)`: số **stepping number** trong `[l, r]` — mọi cặp chữ số kề chênh **đúng** `1` (số một chữ số cũng tính).

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_build_ways(k, maxlen):
    # TODO 1: ways = bảng (maxlen+1) x 10 toàn 0; ways[1][d] = 1
    # TODO 2: với m từ 2 tới maxlen: ways[m][d] = tổng ways[m-1][e] với |d-e| <= k
    pass

# Bài 4 — TODO
def my_count_le(N, k):
    # TODO: ngắn hơn N, rồi từng vị trí i: chọn d < N[i], dừng nếu tiền tố xấu, +1 nếu N tốt
    pass

# Bài 5 — TODO
def count_stepping(l, r):
    # TODO: như bài này nhưng điều kiện |d - e| == 1
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: xâu `3x` với `x` là chữ số cuối.
- Mức 2: `|3 − x| ≤ 1` ⇒ `x ∈ {2, 3, 4}`.
- Mức 3: `3` xâu: `32, 33, 34`.

**Bài 2.**
- Mức 1: `k = 0` ⇒ hai chữ số phải giống nhau.
- Mức 2: các số `10..20`: chỉ `11` có hai chữ số giống nhau.
- Mức 3: đáp án `1`.

**Bài 3.**
- Mức 1: `ways[1][d] = 1` với mọi `d`.
- Mức 2: duyệt `m` tăng dần.
- Mức 3: `sum(ways[m-1][e] for e in range(10) if abs(d-e) <= k)`.

**Bài 4.**
- Mức 1: tách các số ngắn hơn `N`.
- Mức 2: với độ dài bằng `N`, mỗi vị trí `i` thử `d < N[i]`.
- Mức 3: kiểm tra `abs(s[i] - prev) > k` để dừng; cuối cùng `+1` nếu không dừng.

**Bài 5.**
- Mức 1: chỉ đổi điều kiện kề.
- Mức 2: `ways[m][d]` cộng `e` với `abs(d-e) == 1`.
- Mức 3: kiểm tra `abs(s[i]-prev) != 1` để dừng.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert build_ways(1, 2)[2][3] == 3
# Bài 2
assert brute_force(10, 20, 0) == 1 == Solution().goodIntegers(10, 20, 0)

# Bài 3
def my_build_ways_ans(k, maxlen):
    ways = [[0] * 10 for _ in range(maxlen + 1)]
    for d in range(10):
        ways[1][d] = 1
    for m in range(2, maxlen + 1):
        for d in range(10):
            ways[m][d] = sum(ways[m - 1][e] for e in range(10) if abs(d - e) <= k)
    return ways

for k in range(10):
    assert my_build_ways_ans(k, 8) == build_ways(k, 8)
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_count_le_ans(N, k):
    if N <= 0:
        return 0
    s = [int(c) for c in str(N)]
    L = len(s)
    ways = my_build_ways_ans(k, L)
    total = sum(sum(ways[length][d] for d in range(1, 10)) for length in range(1, L))
    prev = -1
    for i in range(L):
        for d in range(1 if i == 0 else 0, s[i]):
            if i == 0 or abs(d - prev) <= k:
                total += ways[L - i][d]
        if i > 0 and abs(s[i] - prev) > k:
            return total
        prev = s[i]
    return total + 1

for N in list(range(0, 300)) + [10**6, 987654321]:
    for k in range(0, 10, 3):
        if N <= 3000:
            assert my_count_le_ans(N, k) == f_brute(N, k), (N, k)
        else:
            assert my_count_le_ans(N, k) == count_le_prefix(N, k, build_ways(k, 10))

# Bài 5: stepping numbers (mọi cặp kề chênh đúng 1)
def count_stepping(l, r):
    L = len(str(r))
    ways = [[0] * 10 for _ in range(L + 1)]
    for d in range(10):
        ways[1][d] = 1
    for m in range(2, L + 1):
        for d in range(10):
            ways[m][d] = sum(ways[m - 1][e] for e in range(10) if abs(d - e) == 1)
    def f(N):
        if N <= 0:
            return 0
        s = [int(c) for c in str(N)]
        n = len(s)
        total = sum(sum(ways[length][d] for d in range(1, 10)) for length in range(1, n))
        prev = -1
        for i in range(n):
            for d in range(1 if i == 0 else 0, s[i]):
                if i == 0 or abs(d - prev) == 1:
                    total += ways[n - i][d]
            if i > 0 and abs(s[i] - prev) != 1:
                return total
            prev = s[i]
        return total + 1
    return f(r) - f(l - 1)

def is_stepping(x):
    d = [int(c) for c in str(x)]
    return all(abs(a - b) == 1 for a, b in zip(d, d[1:]))

for l, r in [(1, 100), (10, 200), (50, 5000), (1, 10**5)]:
    assert count_stepping(l, r) == sum(1 for x in range(l, r + 1) if is_stepping(x)), (l, r)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đếm số `≤ N` thỏa điều kiện kề" bằng bảng `ways`:**

```python
ways[1][d] = 1
ways[m][d] = sum(ways[m-1][e] for e in range(10) if ĐIỀU_KIỆN(d, e))

def f(N):                                    # số số thỏa trong [1, N]
    s = list(map(int, str(N))); n = len(s)
    total = sum(sum(ways[ln][d] for d in range(1, 10)) for ln in range(1, n))   # ngắn hơn
    prev = -1
    for i in range(n):
        for d in range(1 if i == 0 else 0, s[i]):
            if i == 0 or ĐIỀU_KIỆN(prev, d):
                total += ways[n - i][d]
        if i > 0 and not ĐIỀU_KIỆN(prev, s[i]):
            return total                      # tiền tố đã xấu
        prev = s[i]
    return total + 1                          # chính N thỏa
answer = f(r) - f(l - 1)
```

**Checklist trước khi submit:**

- [ ] `ways[1][d] = 1` cho mọi `d` (số một chữ số luôn tốt).
- [ ] Chữ số đầu của số có độ dài `L` bắt đầu từ `1` (không `0`).
- [ ] Các số **ngắn hơn** `N` đã cộng riêng.
- [ ] Dừng ngay khi `N[0..i]` có cặp xấu; `+1` chỉ khi đi hết mà không dừng.
- [ ] Hai đầu: `f(r) − f(l − 1)`.
- [ ] `k = 0` và `k = 9` kiểm tra bằng assert.
- [ ] Đã đối chiếu vét cạn trên đoạn nhỏ ngẫu nhiên.